# plotmux — backend examples

This notebook shows how to use `plotmux`'s public API (`plotmux.hist`, `plotmux.line`, `plotmux.scatter`, `plotmux.layer`) with a given backend. It only uses backend-agnostic features, so it works unchanged with any registered backend.

To try a different backend, change `BACKEND` in the cell below and re-run the notebook — the rest of the notebook does not need to change.

In [ ]:
# Select the backend to use throughout this notebook.
# Available backends: "matplotlib", "xy", "bokeh", "altair".
BACKEND = "matplotlib"

In [ ]:
import numpy as np

import plotmux

plotmux.set_backend(BACKEND)

rng = np.random.default_rng(42)
values = rng.normal(loc=0.0, scale=1.0, size=100_000)
x = np.linspace(0.0, 10.0, 200)
y = np.sin(x)
scatter_x = rng.uniform(0.0, 10.0, size=200)
scatter_y = scatter_x + rng.normal(scale=0.5, size=200)

## Basic histogram

`plotmux.hist` builds a `HistogramSpec` and renders it with the current default backend (set above via `plotmux.set_backend`).

In [ ]:
fig = plotmux.hist(values, bins=101)
fig.backend_name

In [ ]:
fig.to_native()

## Custom bin count and axis range

`xmin`/`xmax` accept explicit values or quantile strings such as `"q0.1"` (10th percentile), resolved via `plotmux.core.range.find_range`.

In [ ]:
fig = plotmux.hist(values, bins=101, xmin="q0.01", xmax="q0.99")
fig.to_native()

## Probability density

Set `density=True` so the histogram integrates to 1 instead of showing raw counts.

In [ ]:
fig = plotmux.hist(values, bins=101, density=True)
fig.to_native()

## Label

Passing `label` attaches a name to the histogram, used e.g. as a legend entry.

In [ ]:
fig = plotmux.hist(values, bins=101, density=True, label="group A")
fig.to_native()

## Color

`color` accepts a hex string, a CSS/matplotlib named color, or an RGB(A) float tuple, normalized internally via `plotmux.colors.parse_color`.

In [ ]:
fig = plotmux.hist(values, bins=101, color="tab:green", label="named")
fig.to_native()

In [ ]:
fig = plotmux.hist(values, bins=101, color="#ff8800", label="hex")
fig.to_native()

In [ ]:
fig = plotmux.hist(values, bins=101, color=(0.2, 0.4, 0.8, 0.7), label="rgba")
fig.to_native()

## Titles and axis labels

`title`, `xlabel`, and `ylabel` are backend-agnostic and forwarded to every backend's native title/label mechanism.

In [ ]:
fig = plotmux.hist(
    values,
    bins=101,
    title="Standard normal sample",
    xlabel="value",
    ylabel="count",
)
fig.to_native()

## Log-scaled axes

`yscale="log"` is useful to inspect the tails of a histogram.

In [ ]:
fig = plotmux.hist(values, bins=101, yscale="log")
fig.to_native()

## Empirical CDF

`plotmux.cdf` plots the empirical cumulative distribution function of an array of values. It accepts the same `xmin`/`xmax`/`color`/`label`/`title`/axis-label arguments as `plotmux.hist`, plus `nbins` to control the binning used to approximate the curve. `ylabel` defaults to `"cumulative probability"`.

In [ ]:
fig = plotmux.cdf(values, nbins=101, title="Standard normal CDF")
fig.to_native()

In [ ]:
fig = plotmux.cdf(
    values, nbins=101, xmin="q0.01", xmax="q0.99", color="tab:purple", label="group A"
)
fig.to_native()

## Line chart

`plotmux.line` plots a simple `x`/`y` line, sharing the same `color`, `label`, `title`, `xlabel`, `ylabel`, `xscale`, and `yscale` arguments as `plotmux.hist`.

In [ ]:
fig = plotmux.line(x, y, label="sin(x)", color="tab:blue")
fig.to_native()

## Scatter chart

`plotmux.scatter` plots `x`/`y` points and additionally accepts a `size` for the markers.

In [ ]:
fig = plotmux.scatter(
    scatter_x,
    scatter_y,
    label="noisy y = x",
    color="tab:red",
    size=20,
)
fig.to_native()

## Layering multiple specs

`plotmux.layer` draws several child specs (or already-rendered `Figure`s) onto one shared axes, e.g. a scatter plot with a fitted line on top.

In [ ]:
coeffs = np.polyfit(scatter_x, scatter_y, deg=1)
fit_x = np.linspace(scatter_x.min(), scatter_x.max(), 50)
fit_y = np.polyval(coeffs, fit_x)

fig = plotmux.layer(
    plotmux.scatter(scatter_x, scatter_y, label="data", color="tab:gray"),
    plotmux.line(fit_x, fit_y, label="fit", color="tab:red"),
    title="Linear fit",
    xlabel="x",
    ylabel="y",
)
fig.to_native()

## Default categorical palette

When a `LayerSpec` child sets no explicit `color`, backends assign successive colors from `plotmux.colors.DEFAULT_PALETTE`.

In [ ]:
fig = plotmux.layer(
    plotmux.hist(rng.normal(-1, 1, 5_000), bins=101, density=True, label="A"),
    plotmux.hist(rng.normal(1, 1, 5_000), bins=101, density=True, label="B"),
    plotmux.hist(rng.normal(0, 2, 5_000), bins=101, density=True, label="C"),
    title="Default palette",
)
fig.to_native()

## Grid layouts

`plotmux.grid` lays out several specs (or already-rendered `Figure`s) as independent panels, the backend-agnostic equivalent of `pyplot.subplots`. Unlike `plotmux.layer`, each item gets its own axes; an item may itself be built with `plotmux.layer(...)`.

In [ ]:
fig = plotmux.grid(
    plotmux.hist(values, bins=101, title="Histogram"),
    plotmux.cdf(values, nbins=101, title="CDF"),
    plotmux.line(x, y, title="Line"),
    plotmux.scatter(scatter_x, scatter_y, title="Scatter"),
    ncols=2,
    title="Four independent panels",
)
fig.to_native()

`Figure.save` infers the export format from the file suffix (`.png`, `.svg`, `.pdf`, ...) and delegates to the backend. `matplotlib` and `xy` support several formats; the cells below use whatever formats the selected `BACKEND` actually supports (see `fig.supported_formats` above).

In [ ]:
fig = plotmux.hist(values, bins=101)
formats = sorted(fig.supported_formats)
formats

In [ ]:
for fmt in formats:
    fig.save(f"../tmp/plotmux_hist.{fmt}")

Different backends support different export formats -- `bokeh` only supports `"html"`, and `altair` only supports `"html"`/`"json"`, since static image export needs an extra, environment-specific dependency for each. `Figure.supported_formats` reports what a given figure accepts, so you can check ahead of time instead of catching a `ValueError` from `save`.

In [ ]:
fig.supported_formats

## Explicit backend selection

The backend can also be selected per call, or scoped with a context manager, instead of relying on the process-wide default set above.

In [ ]:
fig = plotmux.hist(values, bins=101, backend=BACKEND)
fig.backend_name

In [ ]:
with plotmux.backend(BACKEND):
    fig = plotmux.hist(values, bins=101)
fig.backend_name